# Fresh Start NBA — Notebook 2: EDA & Visualization

Explore your data and model performance visually.
Run **01_data_setup.ipynb** first so the paths are configured.

What's in this notebook:
- Stat distributions (pts, trb, ast)
- Prediction vs actual scatter plots (loaded from picks_history)
- Model accuracy breakdown by stat
- Bias correction visualization
- Confidence calibration curve
- Feature importance (SHAP values from your trained XGBoost models)

In [ ]:
# ── Re-run paths from notebook 1 ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR   = Path('/content/drive/MyDrive/Fresh_Start_NBA_Colab')
DATA_DIR   = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
OUT_DIR    = BASE_DIR / 'outputs'

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

nba   = pd.read_csv(DATA_DIR / 'nba_data.csv', low_memory=False)
nba['game_date'] = pd.to_datetime(nba['game_date'])
print(f'Loaded nba_data: {len(nba):,} rows')

## 1 — Stat Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
stats = ['pts', 'trb', 'ast', 'stl', 'blk', 'tov']
colors = sns.color_palette('muted', 6)

for ax, stat, color in zip(axes.flat, stats, colors):
    data = nba[stat].dropna()
    ax.hist(data, bins=40, color=color, edgecolor='white', linewidth=0.4)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'mean={data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle=':', linewidth=1.5, label=f'median={data.median():.1f}')
    ax.set_title(stat.upper(), fontsize=13, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('NBA Player Stat Distributions (all seasons)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'stat_distributions.png', bbox_inches='tight')
plt.show()

## 2 — Model Performance Summary (from results.json)

In [ ]:
with open(MODELS_DIR / 'results.json') as f:
    results = json.load(f)

with open(MODELS_DIR / 'bias.json') as f:
    bias = json.load(f)

stats_tracked = list(results.keys())
maes = [results[s]['mae'] for s in stats_tracked]
accs = [results[s]['accuracy'] * 100 for s in stats_tracked]
biases = [bias.get(s, 0) for s in stats_tracked]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MAE bar chart
bars = axes[0].bar(stats_tracked, maes, color=sns.color_palette('Blues_d', len(stats_tracked)))
axes[0].set_title('Mean Absolute Error by Stat', fontweight='bold')
axes[0].set_ylabel('MAE (stat units)')
for bar, val in zip(bars, maes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.2f}',
                 ha='center', va='bottom', fontsize=10)

# OVER/UNDER accuracy bar chart
colors_acc = ['#2ecc71' if a >= 60 else '#e67e22' if a >= 55 else '#e74c3c' for a in accs]
bars2 = axes[1].bar(stats_tracked, accs, color=colors_acc)
axes[1].axhline(50, color='black', linestyle='--', linewidth=1, label='Coin flip (50%)')
axes[1].set_title('OVER/UNDER Accuracy by Stat', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 80)
axes[1].legend()
for bar, val in zip(bars2, accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%',
                 ha='center', va='bottom', fontsize=10)

# Bias chart
colors_bias = ['#e74c3c' if b < 0 else '#2ecc71' for b in biases]
bars3 = axes[2].bar(stats_tracked, biases, color=colors_bias)
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set_title('Model Bias (applied correction)', fontweight='bold')
axes[2].set_ylabel('Bias offset (stat units)')
for bar, val in zip(bars3, biases):
    ypos = bar.get_height() + 0.005 if val >= 0 else bar.get_height() - 0.015
    axes[2].text(bar.get_x() + bar.get_width()/2, ypos, f'{val:+.3f}',
                 ha='center', va='bottom', fontsize=10)

plt.suptitle('XGBoost Model Performance Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'model_performance_summary.png', bbox_inches='tight')
plt.show()

print('\nRaw numbers:')
summary_df = pd.DataFrame({'stat': stats_tracked, 'mae': maes, 'accuracy_%': accs, 'bias': biases})
print(summary_df.to_string(index=False))

## 3 — Picks History: Prediction vs Actual

In [ ]:
# Load picks history if available on Drive
picks_path = BASE_DIR / 'data' / 'picks_history.csv'
if not picks_path.exists():
    # Try outputs folder
    picks_path = BASE_DIR / 'outputs' / 'picks_history.csv'

if picks_path.exists():
    picks = pd.read_csv(picks_path, low_memory=False)
    print(f'Loaded picks_history: {len(picks):,} rows')
    print(f'Columns: {list(picks.columns)}')
    picks.head(3)
else:
    print('picks_history.csv not found on Drive.')
    print('Upload it to Fresh_Start_NBA_Colab/data/ or outputs/ and re-run this cell.')
    picks = None

In [ ]:
if picks is not None and 'result' in picks.columns:
    graded = picks[picks['result'].isin(['WIN', 'LOSS', 'PUSH'])].copy()
    print(f'Graded picks: {len(graded):,}')

    # Win rate by prop type
    if 'prop' in graded.columns:
        wr_by_prop = graded.groupby('prop').apply(
            lambda x: pd.Series({
                'n': len(x),
                'wins': (x['result'] == 'WIN').sum(),
                'win_rate': (x['result'] == 'WIN').mean()
            })
        ).reset_index()
        wr_by_prop = wr_by_prop[wr_by_prop['n'] >= 5].sort_values('win_rate', ascending=False)

        fig, ax = plt.subplots(figsize=(10, 5))
        colors = ['#2ecc71' if w >= 0.55 else '#e67e22' if w >= 0.50 else '#e74c3c'
                  for w in wr_by_prop['win_rate']]
        bars = ax.bar(wr_by_prop['prop'], wr_by_prop['win_rate'] * 100, color=colors)
        ax.axhline(50, color='black', linestyle='--', linewidth=1.2, label='Break-even (50%)')
        ax.set_title('Pick Win Rate by Prop Type', fontsize=13, fontweight='bold')
        ax.set_ylabel('Win Rate (%)')
        ax.set_ylim(0, 80)
        ax.legend()
        for bar, row in zip(bars, wr_by_prop.itertuples()):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{row.win_rate*100:.1f}%\n(n={int(row.n)})',
                    ha='center', va='bottom', fontsize=9)
        plt.tight_layout()
        plt.savefig(OUT_DIR / 'win_rate_by_prop.png', bbox_inches='tight')
        plt.show()
        print(wr_by_prop.to_string(index=False))
else:
    print('Skipping — no graded picks found.')

## 4 — Feature Importance (top 20 features per stat)

In [ ]:
# Load feature importance JSON
fi_path = MODELS_DIR / 'feature_importance_advanced.json'

if fi_path.exists():
    with open(fi_path) as f:
        fi_data = json.load(f)

    stats_to_plot = ['pts', 'trb', 'ast', 'pa']
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))

    for ax, stat in zip(axes.flat, stats_to_plot):
        if stat not in fi_data:
            ax.set_title(f'{stat.upper()} — no data')
            continue

        fi_stat = fi_data[stat]
        # fi_stat should be a dict {feature: importance} or list of [feature, importance]
        if isinstance(fi_stat, dict):
            fi_df = pd.DataFrame(list(fi_stat.items()), columns=['feature', 'importance'])
        else:
            fi_df = pd.DataFrame(fi_stat, columns=['feature', 'importance'])

        fi_df = fi_df.sort_values('importance', ascending=False).head(20)

        colors_fi = sns.color_palette('viridis', len(fi_df))
        ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color=colors_fi)
        ax.set_title(f'{stat.upper()} — Top 20 Features', fontsize=12, fontweight='bold')
        ax.set_xlabel('Feature Importance (XGBoost gain)')

    plt.suptitle('XGBoost Feature Importance by Stat', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'feature_importance.png', bbox_inches='tight')
    plt.show()
else:
    print(f'feature_importance_advanced.json not found at {fi_path}')
    print('Train models first (notebook 4) to generate this file.')

## 5 — SHAP Values (deep feature attribution)

In [ ]:
# Install SHAP if not available
try:
    import shap
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'shap', '-q'])
    import shap

print('SHAP version:', shap.__version__)

In [ ]:
import json, pickle

# Load feature column list
with open(MODELS_DIR.parent / 'feature_cols_advanced.json') as f:
    feat_meta = json.load(f)
FEATURE_COLS = feat_meta['feature_columns']

# Pick a stat to analyze — change this to explore others
STAT_TO_ANALYZE = 'pts'   # options: pts, trb, ast, pa, pr, pra, tov

model_path = MODELS_DIR / f'xgb_{STAT_TO_ANALYZE}_advanced.pkl'
if model_path.exists():
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    print(f'Loaded model: xgb_{STAT_TO_ANALYZE}_advanced.pkl')
else:
    print(f'Model not found: {model_path}')
    model = None

In [ ]:
if model is not None:
    # Build a sample of feature-engineered rows
    # We'll use a rolling-average approximation since full feature pipeline isn't run here
    stats_base = ['pts', 'trb', 'ast', 'stl', 'blk', 'tov', 'mp', 'fga', 'fta', '3pa', '3p_pct', 'fg_pct', 'ft_pct']
    players = nba.sort_values('game_date').groupby('player').tail(30)

    avail_feats = [c for c in FEATURE_COLS if c in nba.columns]
    sample = players[avail_feats].dropna().head(500)

    if len(sample) > 0:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(sample)

        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, sample, plot_type='bar',
                          max_display=20, show=False)
        plt.title(f'SHAP Feature Importance — {STAT_TO_ANALYZE.upper()}', fontweight='bold')
        plt.tight_layout()
        plt.savefig(OUT_DIR / f'shap_{STAT_TO_ANALYZE}.png', bbox_inches='tight')
        plt.show()

        plt.figure(figsize=(10, 10))
        shap.summary_plot(shap_values, sample, max_display=20, show=False)
        plt.title(f'SHAP Dot Plot — {STAT_TO_ANALYZE.upper()}', fontweight='bold')
        plt.tight_layout()
        plt.savefig(OUT_DIR / f'shap_dot_{STAT_TO_ANALYZE}.png', bbox_inches='tight')
        plt.show()
    else:
        print('Not enough overlapping features between model and raw data for SHAP analysis.')
        print('Run full feature pipeline (notebook 3) first for complete SHAP analysis.')

## 6 — Player Trend: Last 20 Games

In [ ]:
# Change the player name to any player in your dataset
PLAYER_NAME = 'Luka Doncic'   # modify this
STAT        = 'pts'            # pts, trb, ast, etc.

player_data = nba[nba['player'].str.lower() == PLAYER_NAME.lower()].copy()

if len(player_data) == 0:
    # Fuzzy match
    matches = nba[nba['player'].str.lower().str.contains(PLAYER_NAME.lower().split()[0])]['player'].unique()
    print(f'Player "{PLAYER_NAME}" not found. Possible matches:')
    print(matches[:10])
else:
    player_data = player_data.sort_values('game_date').tail(30)

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Stat line
    axes[0].plot(player_data['game_date'], player_data[STAT], 'o-', color='steelblue', linewidth=2, markersize=5)
    rolling_avg = player_data[STAT].rolling(5).mean()
    axes[0].plot(player_data['game_date'], rolling_avg, '--', color='red', linewidth=1.5, alpha=0.8, label='L5 avg')
    axes[0].set_title(f'{PLAYER_NAME} — {STAT.upper()} (last 30 games)', fontsize=13, fontweight='bold')
    axes[0].set_ylabel(STAT.upper())
    axes[0].legend()

    # Minutes
    axes[1].bar(player_data['game_date'], player_data['mp'], color='steelblue', alpha=0.6)
    axes[1].set_title('Minutes Played', fontsize=11)
    axes[1].set_ylabel('MP')
    axes[1].set_xlabel('Game Date')

    plt.tight_layout()
    plt.savefig(OUT_DIR / f'player_trend_{PLAYER_NAME.replace(" ", "_")}_{STAT}.png', bbox_inches='tight')
    plt.show()

    recent = player_data[STAT]
    print(f'\n{PLAYER_NAME} {STAT.upper()} recent stats:')
    print(f'  L5 avg:  {recent.tail(5).mean():.1f}')
    print(f'  L10 avg: {recent.tail(10).mean():.1f}')
    print(f'  L20 avg: {recent.tail(20).mean():.1f}')
    print(f'  Std dev: {recent.tail(10).std():.2f}')